<a href="https://colab.research.google.com/github/AdAdalan/NLP-Assignment3/blob/main/notebooks/Junyao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

%cd /content

if not os.path.exists("/content/NLP-Assignment3"):
  !git clone https://github.com/AdAdalan/NLP-Assignment3.git
else:
  print("Repo already exists.")

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATA_DIR = Path("/content/drive/MyDrive/NLP ASS3/data")

/content
Repo already exists.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json

#read data
with open(f"{DATA_DIR}/train-claims.json", "r") as f:
    train_claims = json.load(f)

with open(f"{DATA_DIR}/dev-claims.json", "r") as f:
    dev_claims = json.load(f)

with open(f"{DATA_DIR}/test-claims-unlabelled.json", "r") as f:
    test_claims = json.load(f)

with open(f"{DATA_DIR}/evidence.json", "r") as f:
    evidence = json.load(f)

print("Train claims length:", len(train_claims))
print("Dev claims length:", len(dev_claims))
print("Test claims length:", len(test_claims))
print("Evidence passages length:", len(evidence))

Train claims length: 1228
Dev claims length: 154
Test claims length: 153
Evidence passages length: 1208827


In [ ]:
from collections import Counter
import numpy as np

# 统计训练集标签分布
label_dist = Counter(c["claim_label"] for c in train_claims.values())

print("Training Set Label Distribution:")
for lbl, cnt in label_dist.most_common():
    print(f"{lbl:20s} {cnt:5d} ({cnt / len(train_claims):.1%})")

# 统计每条 claim 对应的 GT evidence 数量
gt_counts = [len(c["evidences"]) for c in train_claims.values()]

print(
    f"\nGround Truth Evidence For Each Claim: "
    f"min={min(gt_counts)}, "
    f"max={max(gt_counts)}, "
    f"mean={np.mean(gt_counts):.2f}, "
    f"median={int(np.median(gt_counts))}"
)

Training Set Label Distribution:
SUPPORTS               519 (42.3%)
NOT_ENOUGH_INFO        386 (31.4%)
REFUTES                199 (16.2%)
DISPUTED               124 (10.1%)

Ground Truth Evidence For Each Claim: min=1, max=5, mean=3.36, median=3


In [ ]:
import random

sample_evidence = random.sample(list(evidence.items()), 20)

for eid, text in sample_evidence:
    print("ID:", eid)
    print(text)
    print("-" * 80)

ID: evidence-580270
The album was composed as a dark, continuous tone poem divided by four sections of Davis ' jazz fusion recordings.
--------------------------------------------------------------------------------
ID: evidence-877477
John Laurence Rentoul (6 July 1846 -- 15 April 1926), was a member of the Presbyterian clergy and a poet.
--------------------------------------------------------------------------------
ID: evidence-49518
A guide to bird tracking has been published.
--------------------------------------------------------------------------------
ID: evidence-661184
Sultangazi has 3 precincts, 11 neighbourhoods and 1 village.
--------------------------------------------------------------------------------
ID: evidence-361713
Modern coach or stagecoach transportation has its origins in the basterna.
--------------------------------------------------------------------------------
ID: evidence-473310
He was described as epitomizing ``the strong link between the 19th century

In [ ]:
sample_claims = random.sample(list(train_claims.items()), 20)

for cid, c in sample_claims:
    print("ID:", cid)
    print(c["claim_text"])
    print("-" * 80)

ID: claim-3105
Global warming theory holds that one of the fingerprints of human-induced global warming is more rapid warming in the lower troposphere than at the surface (James Taylor)
--------------------------------------------------------------------------------
ID: claim-2961
Cooks ’97% consensus’ disproven by a new peer
--------------------------------------------------------------------------------
ID: claim-399
Climate Change ‘Heat Records’ Are a Huge Data Manipulation
--------------------------------------------------------------------------------
ID: claim-1066
[…] in fact this pattern is already emerging, with the conditions that create extremely warm dry years and extremely wet years both becoming more frequent.
--------------------------------------------------------------------------------
ID: claim-1584
Renewables can't provide baseload power.
--------------------------------------------------------------------------------
ID: claim-3134
Over the last decade, heatwaves a

# Retrival TFI IDF Baseline

In [ ]:
# Baseline TFIDF

import nltk
nltk.download("stopwords")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from nltk.corpus import stopwords


def tfidf(evidences, claims):
    vectorizer = TfidfVectorizer(
        stop_words=stopwords.words("english")
    )

    evidence_tfidf = vectorizer.fit_transform(evidences)
    claims_tfidf = vectorizer.transform(claims)

    cos_similarity = cosine_similarity(claims_tfidf, evidence_tfidf)

    return cos_similarity

def choose_top_k(cos_similarity, evidence_ids, claim_ids, top_k):
    predict_tfidf = {}

    for i, cid in enumerate(claim_ids):
        cos_row = cos_similarity[i]

        # 找出当前 claim 相似度最高的 top_k 个 evidence 的 index
        top_idx = np.argpartition(-cos_row, top_k)[:top_k]  # O(n)

        # 根据 index 找回 evidence_id
        predict_tfidf[cid] = [evidence_ids[idx] for idx in top_idx]

    return predict_tfidf

def eval_retrieval(claims_dataset, predict):
    all_recalls = []
    all_precisions = []
    all_fscores = []

    for claim_id, claim in sorted(claims_dataset.items()):

        if claim_id not in predict:
            continue

        evidence_correct = 0
        evidence_recall = 0.0
        evidence_precision = 0.0
        evidence_fscore = 0.0

        if isinstance(predict[claim_id], list) and len(predict[claim_id]) > 0:
            predict_set = set(predict[claim_id])

            for true_id in claim["evidences"]:
                if true_id in predict_set:
                    evidence_correct += 1

            if evidence_correct > 0:
                evidence_recall = evidence_correct / len(claim["evidences"])
                evidence_precision = evidence_correct / len(predict[claim_id])
                evidence_fscore = (
                    2 * evidence_precision * evidence_recall
                ) / (
                    evidence_precision + evidence_recall
                )

        all_recalls.append(evidence_recall)
        all_precisions.append(evidence_precision)
        all_fscores.append(evidence_fscore)

        min_recall =min(all_recalls)


    print(f"Mean Recall:    {np.mean(all_recalls):.6f}")
    print(f"Mean Precision: {np.mean(all_precisions):.6f}")
    print(f"Mean F1-Score:  {np.mean(all_fscores):.6f}")
    print(f"Min Recall:     {min_recall:.6f}")

    return np.mean(all_fscores), np.mean(all_precisions), np.mean(all_recalls), min_recall

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
evidence_list_ids = list(evidence.keys())
evidence_list_texts = list(evidence.values())
dev_list_ids = list(dev_claims.keys())
dev_list_claims = list(dev_claims[id]["claim_text"] for id in dev_list_ids)

print(evidence_list_ids[3])
print(evidence_list_texts[3])
print(dev_list_ids[3])
print(dev_list_claims[3])

evidence-3
Gerald Francis Goyer (born October 20, 1936) was a professional ice hockey player who played 40 games in the National Hockey League.
claim-871
“As it happens, Zika may also be a good model of the second worrying effect — disease mutation.


In [ ]:
Top_K = 5
cos_row = tfidf(evidence_list_texts, dev_list_claims)
predict_R1 = choose_top_k(cos_row, evidence_list_ids, dev_list_ids, Top_K)
evaluation = eval_retrieval(dev_claims, predict_R1)


Mean Recall:    0.148701
Mean Precision: 0.074026
Mean F1-Score:  0.091548
Min Recall:     0.000000


# BM25S

In [22]:
!pip install -q bm25s sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 6.4 MB/s eta 0:00:00


In [23]:
import time
import bm25s
import psutil

evidence_ids = list(evidence.keys())
evidence_texts = [evidence[eid] for eid in evidence_ids]


print("Tokenizing 1.2M evidence ...")
t0 = time.time()

corpus_tokens = bm25s.tokenize(
    evidence_texts,
    stopwords="en",
    stemmer=None
)

print(f"    tokenize 用时: {time.time() - t0:.0f}s")

print("\nbuilding BM25 index ...")
t0 = time.time()

retriever = bm25s.BM25()
retriever.index(corpus_tokens)

print(f"    index 用时: {time.time() - t0:.0f}s")

Tokenizing 1.2M evidence ...


Split strings:   0%|          | 0/1208827 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


    tokenize 用时: 19s

building BM25 index ...


BM25S Count Tokens:   0%|          | 0/1208827 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/1208827 [00:00<?, ?it/s]

    index 用时: 32s


In [26]:
from tqdm import tqdm

def bm25s_retrieve(claims_dict, k=100):
    cids = list(claims_dict.keys())
    texts = [claims_dict[c]["claim_text"] for c in cids]

    query_tokens = bm25s.tokenize(
        texts,
        stopwords="en",
        stemmer=None
    )

    results, scores = retriever.retrieve(
        query_tokens,
        k=k
    )

    out = {}
    for i, cid in enumerate(cids):
        out[cid] = [evidence_ids[j] for j in results[i]]

    return out

In [27]:
print("BM25s baseline(top-5)")
TOP_K = 5
t0 = time.time()

predict_R2 = bm25s_retrieve(
    dev_claims,
    k=TOP_K
)

print(f"用时: {time.time() - t0:.1f}s\n")

f_R2 = eval_retrieval(
    dev_claims,
    predict_R2
)

BM25s baseline(top-5)


Split strings:   0%|          | 0/154 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/154 [00:00<?, ?it/s]

用时: 1.0s

Mean Recall:    0.160281
Mean Precision: 0.089610
Mean F1-Score:  0.107689
Min Recall:     0.000000


# pretrained model

In [ ]:
import torch
!pip install -q transformers
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

In [ ]:
evidence_ids = list(evidence.keys())
evidence_texts = [evidence[eid] for eid in evidence_ids]

dev_ids = list(dev_claims.keys())
dev_texts = [dev_claims[cid]["claim_text"] for cid in dev_ids]

print("Num evidence:", len(evidence_texts))
print("Num dev claims:", len(dev_texts))

Num evidence: 1208827
Num dev claims: 154


# BGE

In [ ]:
BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

bge_tokenizer = AutoTokenizer.from_pretrained(BGE_MODEL_NAME)
bge_model = AutoModel.from_pretrained(BGE_MODEL_NAME).to(device)
bge_model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
def encode_bge_texts(texts, tokenizer, model, batch_size=64, max_len=256):
    all_embeddings = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Encoding with BGE"):
            batch_texts = texts[i:i + batch_size]

            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt"
            )

            encoded = {
                k: v.to(device)
                for k, v in encoded.items()
            }

            outputs = model(**encoded)

            # CLS pooling: [batch_size, hidden_dim]
            embeddings = outputs.last_hidden_state[:, 0]

            # normalize，方便之后直接用 dot product 当 cosine similarity
            embeddings = torch.nn.functional.normalize(
                embeddings,
                p=2,
                dim=1
            )

            all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings, dim=0)

对 BGE-small-en-v1.5 来说，更标准的 query instruction:

In [ ]:
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

evidence_ids = list(evidence.keys())
evidence_texts = [evidence[eid] for eid in evidence_ids]

dev_ids = list(dev_claims.keys())
dev_texts = [dev_claims[cid]["claim_text"] for cid in dev_ids]

bge_evidence_inputs = evidence_texts

bge_dev_inputs = [
    BGE_QUERY_PREFIX + text
    for text in dev_texts
]

In [ ]:
bge_evidence_emb = encode_bge_texts(
    bge_evidence_inputs,
    bge_tokenizer,
    bge_model,
    batch_size=64,
    max_len=256
)

bge_dev_emb = encode_bge_texts(
    bge_dev_inputs,
    bge_tokenizer,
    bge_model,
    batch_size=64,
    max_len=256
)

print("BGE evidence emb:", bge_evidence_emb.shape)
print("BGE dev emb:", bge_dev_emb.shape)

Encoding with BGE: 100%|██████████| 3/3 [00:00<00:00, 12.34it/s]

BGE evidence emb: torch.Size([1208827, 384])
BGE dev emb: torch.Size([154, 384])


改成 Google Drive

In [ ]:
import pickle

SAVE_DIR = "/content/drive/MyDrive/NLP ASS3/saved_bge_embeddings"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(bge_evidence_emb, f"{SAVE_DIR}/bge_evidence_emb.pt")
torch.save(bge_dev_emb, f"{SAVE_DIR}/bge_dev_emb.pt")

with open(f"{SAVE_DIR}/evidence_ids.pkl", "wb") as f:
    pickle.dump(evidence_ids, f)

with open(f"{SAVE_DIR}/dev_ids.pkl", "wb") as f:
    pickle.dump(dev_ids, f)

print("Saved BGE embeddings to Google Drive.")

Saved BGE embeddings to Google Drive.


In [ ]:
import os
import torch
import pickle

SAVE_DIR = "/content/drive/MyDrive/NLP ASS3/saved_bge_embeddings"

bge_evidence_emb = torch.load(f"{SAVE_DIR}/bge_evidence_emb.pt")
bge_dev_emb = torch.load(f"{SAVE_DIR}/bge_dev_emb.pt")

with open(f"{SAVE_DIR}/evidence_ids.pkl", "rb") as f:
    evidence_ids = pickle.load(f)

with open(f"{SAVE_DIR}/dev_ids.pkl", "rb") as f:
    dev_ids = pickle.load(f)

print("Loaded BGE embeddings from Google Drive.")
print("bge_evidence_emb:", bge_evidence_emb.shape)
print("bge_dev_emb:", bge_dev_emb.shape)
print("evidence_ids:", len(evidence_ids))
print("dev_ids:", len(dev_ids))

Loaded BGE embeddings from Google Drive.
bge_evidence_emb: torch.Size([1208827, 384])
bge_dev_emb: torch.Size([154, 384])
evidence_ids: 1208827
dev_ids: 154


In [ ]:
import torch
import pickle
import os

SAVE_DIR = "/content/drive/MyDrive/NLP ASS3/saved_bge_embeddings"

print("SAVE_DIR exists:", os.path.exists(SAVE_DIR))
print("Files:", os.listdir(SAVE_DIR))

bge_evidence_emb = torch.load(f"{SAVE_DIR}/bge_evidence_emb.pt")
bge_dev_emb = torch.load(f"{SAVE_DIR}/bge_dev_emb.pt")

with open(f"{SAVE_DIR}/evidence_ids.pkl", "rb") as f:
    evidence_ids = pickle.load(f)

with open(f"{SAVE_DIR}/dev_ids.pkl", "rb") as f:
    dev_ids = pickle.load(f)

print("BGE evidence emb:", bge_evidence_emb.shape)
print("BGE dev emb:", bge_dev_emb.shape)
print("Num evidence ids:", len(evidence_ids))
print("Num dev ids:", len(dev_ids))

SAVE_DIR exists: True
Files: ['bge_evidence_emb.pt', 'bge_dev_emb.pt', 'evidence_ids.pkl', 'dev_ids.pkl']
BGE evidence emb: torch.Size([1208827, 384])
BGE dev emb: torch.Size([154, 384])
Num evidence ids: 1208827
Num dev ids: 154


In [ ]:
def dense_retrieve_from_embeddings(
    claim_ids,
    claim_emb,
    evidence_ids,
    evidence_emb,
    top_k=100,
    batch_size=32
):
    """
    claim_emb: [num_claims, dim]
    evidence_emb: [num_evidence, dim]
    两边都已经 normalize，所以 dot product = cosine similarity
    """
    predict = {}

    evidence_emb_t = evidence_emb.T  # [dim, num_evidence]

    for start in tqdm(range(0, len(claim_ids), batch_size), desc="Dense retrieving"):
        end = start + batch_size

        batch_claim_emb = claim_emb[start:end]  # [batch, dim]

        scores = batch_claim_emb @ evidence_emb_t  # [batch, num_evidence]

        top_scores, top_indices = torch.topk(
            scores,
            k=top_k,
            dim=1
        )

        top_indices = top_indices.cpu().numpy()

        for i, cid in enumerate(claim_ids[start:end]):
            predict[cid] = [
                evidence_ids[idx]
                for idx in top_indices[i]
            ]

    return predict

In [ ]:
TOP_K = 5

predict_R3_bge = dense_retrieve_from_embeddings(
    claim_ids=dev_ids,
    claim_emb=bge_dev_emb,
    evidence_ids=evidence_ids,
    evidence_emb=bge_evidence_emb,
    top_k=TOP_K,
    batch_size=32
)

Dense retrieving: 100%|██████████| 5/5 [00:03<00:00,  1.36it/s]


In [17]:
def eval_retrieval(claims_dataset, predict):
    all_recalls = []
    all_precisions = []
    all_fscores = []

    for claim_id, claim in sorted(claims_dataset.items()):

        if claim_id not in predict:
            continue

        evidence_correct = 0
        evidence_recall = 0.0
        evidence_precision = 0.0
        evidence_fscore = 0.0

        if isinstance(predict[claim_id], list) and len(predict[claim_id]) > 0:
            predict_set = set(predict[claim_id])

            for true_id in claim["evidences"]:
                if true_id in predict_set:
                    evidence_correct += 1

            if evidence_correct > 0:
                evidence_recall = evidence_correct / len(claim["evidences"])
                evidence_precision = evidence_correct / len(predict[claim_id])
                evidence_fscore = (
                    2 * evidence_precision * evidence_recall
                ) / (
                    evidence_precision + evidence_recall
                )

        all_recalls.append(evidence_recall)
        all_precisions.append(evidence_precision)
        all_fscores.append(evidence_fscore)

        min_recall =min(all_recalls)


    print(f"Mean Recall:    {np.mean(all_recalls):.6f}")
    print(f"Mean Precision: {np.mean(all_precisions):.6f}")
    print(f"Mean F1-Score:  {np.mean(all_fscores):.6f}")
    print(f"Min Recall:     {min_recall:.6f}")

    return np.mean(all_fscores), np.mean(all_precisions), np.mean(all_recalls), min_recall

In [ ]:
print("BGE dense retrieval")
f_R3_bge = eval_retrieval(
    dev_claims,
    predict_R3_bge
)

BGE dense retrieval
Mean Recall:    0.240260
Mean Precision: 0.127273
Mean F1-Score:  0.153577
Min Recall:     0.000000


# BM25 top-100 和 BGE top-100 candidates

In [29]:
# ============================================================
# Cell 33: Build larger candidate pools from BM25 and BGE
# ============================================================

CANDIDATE_K = 100

print("Retrieving BM25 top-100 candidates...")
predict_R2_bm25_top100 = bm25s_retrieve(
    dev_claims,
    k=CANDIDATE_K
)

print("Retrieving BGE top-100 candidates...")
predict_R3_bge_top100 = dense_retrieve_from_embeddings(
    claim_ids=dev_ids,
    claim_emb=bge_dev_emb,
    evidence_ids=evidence_ids,
    evidence_emb=bge_evidence_emb,
    top_k=CANDIDATE_K,
    batch_size=32
)

print("Done.")
print("Example claim:", dev_ids[0])
print("BM25 candidates:", len(predict_R2_bm25_top100[dev_ids[0]]))
print("BGE candidates:", len(predict_R3_bge_top100[dev_ids[0]]))

Retrieving BM25 top-100 candidates...


Split strings:   0%|          | 0/154 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/154 [00:00<?, ?it/s]

Retrieving BGE top-100 candidates...


NameError: name 'bge_dev_emb' is not defined

In [ ]:
# ============================================================
# Cell 34: Evaluate candidate recall upper bound
# ============================================================

print("BM25 top-100 candidate performance")
bm25_100_score = eval_retrieval(
    dev_claims,
    predict_R2_bm25_top100
)

print("\nBGE top-100 candidate performance")
bge_100_score = eval_retrieval(
    dev_claims,
    predict_R3_bge_top100
)

BM25 top-100 candidate performance
Mean Recall:    0.439177
Mean Precision: 0.013052
Mean F1-Score:  0.025157
Min Recall:     0.000000

BGE top-100 candidate performance
Mean Recall:    0.570563
Mean Precision: 0.016948
Mean F1-Score:  0.032681
Min Recall:     0.000000


In [ ]:
# ============================================================
# Cell 35: Simple hybrid candidate union
# ============================================================

def union_candidates(bm25_pred, bge_pred, final_k=200):
    hybrid = {}

    for cid in bm25_pred.keys():
        merged = []

        for eid in bm25_pred[cid]:
            if eid not in merged:
                merged.append(eid)

        for eid in bge_pred[cid]:
            if eid not in merged:
                merged.append(eid)

        hybrid[cid] = merged[:final_k]

    return hybrid


predict_R4_union_top200 = union_candidates(
    predict_R2_bm25_top100,
    predict_R3_bge_top100,
    final_k=200
)

print("Hybrid union top-200 candidate performance")
union_200_score = eval_retrieval(
    dev_claims,
    predict_R4_union_top200
)

Hybrid union top-200 candidate performance
Mean Recall:    0.673268
Mean Precision: 0.011036
Mean F1-Score:  0.021629
Min Recall:     0.000000


In [ ]:
# ============================================================
# Cell 36: Reciprocal Rank Fusion for BM25 + BGE
# ============================================================

def reciprocal_rank_fusion(result_dicts, weights=None, rrf_k=60, final_k=100):
    """
    result_dicts: list of retrieval prediction dicts
                  each dict: {claim_id: [evidence_id_1, evidence_id_2, ...]}
    weights: optional weights for each retriever
    rrf_k: smoothing constant, commonly 60
    final_k: number of fused candidates returned
    """
    if weights is None:
        weights = [1.0] * len(result_dicts)

    fused = {}

    claim_ids = list(result_dicts[0].keys())

    for cid in claim_ids:
        scores = {}

        for ridx, result_dict in enumerate(result_dicts):
            ranked_eids = result_dict[cid]
            weight = weights[ridx]

            for rank, eid in enumerate(ranked_eids):
                scores[eid] = scores.get(eid, 0.0) + weight / (rrf_k + rank + 1)

        ranked_items = sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )

        fused[cid] = [eid for eid, score in ranked_items[:final_k]]

    return fused

In [ ]:
# ============================================================
# Cell 37: Fine-grained tuning of BGE-heavy RRF weights
# ============================================================

rrf_settings_fine = []

# Keep BM25 fixed at 1.0, gradually increase BGE weight
for bge_w in [1.0, 1.2, 1.5, 1.8, 2.0, 2.5, 3.0, 4.0, 5.0]:
    rrf_settings_fine.append((1.0, bge_w))

# Also test whether slightly reducing BM25 helps
for bm25_w in [0.3, 0.5, 0.7, 0.8]:
    for bge_w in [1.5, 2.0, 2.5, 3.0, 4.0]:
        rrf_settings_fine.append((bm25_w, bge_w))

best_R4_rrf_top100 = None
best_R4_rrf_top5 = None
best_rrf_weights = None
best_rrf_f1 = -1
best_rrf_precision = None
best_rrf_recall = None
best_rrf_min_recall = None

rrf_results = []

for bm25_w, bge_w in rrf_settings_fine:
    rrf_top100 = reciprocal_rank_fusion(
        [predict_R2_bm25_top100, predict_R3_bge_top100],
        weights=[bm25_w, bge_w],
        rrf_k=60,
        final_k=100
    )

    rrf_top5 = {
        cid: eids[:5]
        for cid, eids in rrf_top100.items()
    }

    f1, precision, recall, min_recall = eval_retrieval(
        dev_claims,
        rrf_top5
    )

    rrf_results.append({
        "bm25_weight": bm25_w,
        "bge_weight": bge_w,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "min_recall": min_recall
    })

    print(
        f"BM25={bm25_w:<4} BGE={bge_w:<4} | "
        f"F1={f1:.6f} | Precision={precision:.6f} | Recall={recall:.6f} | Min Recall={min_recall:.6f}"
    )

    if f1 > best_rrf_f1:
        best_rrf_f1 = f1
        best_rrf_precision = precision
        best_rrf_recall = recall
        best_rrf_min_recall = min_recall
        best_rrf_weights = (bm25_w, bge_w)
        best_R4_rrf_top100 = rrf_top100
        best_R4_rrf_top5 = rrf_top5

print("\nBest fine-grained RRF weights:", best_rrf_weights)
print("Best RRF top-5 F1:", best_rrf_f1)
print("Best RRF top-5 Precision:", best_rrf_precision)
print("Best RRF top-5 Recall:", best_rrf_recall)
print("Best RRF top-5 Min Recall:", best_rrf_min_recall)

Mean Recall:    0.247944
Mean Precision: 0.135065
Mean F1-Score:  0.161827
Min Recall:     0.000000
BM25=1.0  BGE=1.0  | F1=0.161827 | Precision=0.135065 | Recall=0.247944 | Min Recall=0.000000
Mean Recall:    0.262771
Mean Precision: 0.146753
Mean F1-Score:  0.174541
Min Recall:     0.000000
BM25=1.0  BGE=1.2  | F1=0.174541 | Precision=0.146753 | Recall=0.262771 | Min Recall=0.000000
Mean Recall:    0.267857
Mean Precision: 0.149351
Mean F1-Score:  0.177876
Min Recall:     0.000000
BM25=1.0  BGE=1.5  | F1=0.177876 | Precision=0.149351 | Recall=0.267857 | Min Recall=0.000000
Mean Recall:    0.271429
Mean Precision: 0.150649
Mean F1-Score:  0.179875
Min Recall:     0.000000
BM25=1.0  BGE=1.8  | F1=0.179875 | Precision=0.150649 | Recall=0.271429 | Min Recall=0.000000
Mean Recall:    0.269264
Mean Precision: 0.149351
Mean F1-Score:  0.178252
Min Recall:     0.000000
BM25=1.0  BGE=2.0  | F1=0.178252 | Precision=0.149351 | Recall=0.269264 | Min Recall=0.000000
Mean Recall:    0.282684
Mean 

In [ ]:
# ============================================================
# Cell 39: Load cross-encoder reranker
# ============================================================

!pip install -q -U sentence-transformers

from sentence_transformers import CrossEncoder
import numpy as np
from tqdm import tqdm
import torch
import gc

RERANKER_NAME = "BAAI/bge-reranker-base"

try:
    reranker = CrossEncoder(
        RERANKER_NAME,
        max_length=512,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    print("Loaded reranker:", RERANKER_NAME)

except Exception as e:
    print("Failed to load BGE reranker. Error:")
    print(e)
    print("Falling back to MiniLM cross-encoder...")

    RERANKER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    reranker = CrossEncoder(
        RERANKER_NAME,
        max_length=512,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    print("Loaded fallback reranker:", RERANKER_NAME)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded reranker: BAAI/bge-reranker-base


In [ ]:
# ============================================================
# Cell 40: Rerank RRF candidates
# ============================================================

def rerank_candidates(
    claims_dict,
    candidate_dict,
    top_k=5,
    batch_size=8
):
    reranked_predictions = {}
    rerank_scores = {}

    for cid in tqdm(claims_dict.keys(), desc="Reranking candidates"):
        claim_text = claims_dict[cid]["claim_text"]
        candidate_eids = candidate_dict[cid]

        pairs = [
            [claim_text, evidence[eid]]
            for eid in candidate_eids
            if eid in evidence
        ]

        valid_eids = [
            eid
            for eid in candidate_eids
            if eid in evidence
        ]

        if len(pairs) == 0:
            reranked_predictions[cid] = []
            rerank_scores[cid] = []
            continue

        scores = reranker.predict(
            pairs,
            batch_size=batch_size,
            show_progress_bar=False
        )

        order = np.argsort(-scores)

        ranked_eids = [
            valid_eids[i]
            for i in order
        ]

        ranked_scores = [
            float(scores[i])
            for i in order
        ]

        reranked_predictions[cid] = ranked_eids[:top_k]
        rerank_scores[cid] = ranked_scores[:top_k]

    return reranked_predictions, rerank_scores

In [ ]:
# ============================================================
# Cell 41: Evaluate R5 = RRF + reranker
# ============================================================

predict_R5_rerank_top5, R5_rerank_scores = rerank_candidates(
    claims_dict=dev_claims,
    candidate_dict=best_R4_rrf_top100,
    top_k=5,
    batch_size=8
)

print("R5: BM25 + BGE + RRF + reranker top-5")
f_R5 = eval_retrieval(
    dev_claims,
    predict_R5_rerank_top5
)

Reranking candidates: 100%|██████████| 154/154 [01:41<00:00,  1.52it/s]

R5: BM25 + BGE + RRF + reranker top-5
Mean Recall:    0.123918
Mean Precision: 0.061039
Mean F1-Score:  0.076453
Min Recall:     0.000000


In [ ]:
# ============================================================
# Cell 43: Summarise retrieval-only results
# ============================================================

retrieval_results = []

def add_result(name, score_tuple):
    f1, precision, recall, min_recall = score_tuple
    retrieval_results.append({
        "method": name,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "min_recall": min_recall
    })

# 你前面 R2/R3 的变量名可能不同，所以这里重新 evaluate 一遍
add_result("R2 BM25 top5", eval_retrieval(dev_claims, predict_R2))
add_result("R3 BGE top5", eval_retrieval(dev_claims, predict_R3_bge))
add_result("R4 BM25+BGE+RRF top5", eval_retrieval(dev_claims, best_R4_rrf_top5))
add_result("R5 BM25+BGE+RRF+Reranker top5", eval_retrieval(dev_claims, predict_R5_rerank_top5))

import pandas as pd

results_df = pd.DataFrame(retrieval_results)
results_df

Mean Recall:    0.160281
Mean Precision: 0.089610
Mean F1-Score:  0.107689
Min Recall:     0.000000
Mean Recall:    0.240260
Mean Precision: 0.127273
Mean F1-Score:  0.153577
Min Recall:     0.000000
Mean Recall:    0.282684
Mean Precision: 0.154545
Mean F1-Score:  0.185101
Min Recall:     0.000000
Mean Recall:    0.123918
Mean Precision: 0.061039
Mean F1-Score:  0.076453
Min Recall:     0.000000


,method,f1,precision,recall,min_recall
0,R2 BM25 top5,0.107689,0.089610,0.160281,0.0
1,R3 BGE top5,0.153577,0.127273,0.240260,0.0
2,R4 BM25+BGE+RRF top5,0.185101,0.154545,0.282684,0.0
3,R5 BM25+BGE+RRF+Reranker top5,0.076453,0.061039,0.123918,0.0


RRF score = BM25_weight × 1 / (k + rank_BM25)
          + BGE_weight  × 1 / (k + rank_BGE)

rrf_k = 60

In [ ]:
# ============================================================
# Remove R5 reranker and free GPU memory
# ============================================================

import gc
import torch

# Delete reranker-related variables if they exist
for var_name in [
    "reranker",
    "predict_R5_rerank_top5",
    "R5_rerank_scores",
    "f_R5"
]:
    if var_name in globals():
        del globals()[var_name]
        print(f"Deleted {var_name}")

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("GPU memory after deleting R5 reranker:")
!nvidia-smi

# 修改Claim

In [ ]:
# ============================================================
# Cell 1: Install packages
# ============================================================

!pip -q install transformers sentencepiece accelerate tqdm

In [ ]:
# ============================================================
# Cell 2: Imports
# ============================================================

import os
import json
import pickle
from pathlib import Path

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
# ============================================================
# Cell 3: Load data
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path("/content/drive/MyDrive/NLP ASS3/data")

with open(DATA_DIR / "train-claims.json", "r") as f:
    train_claims = json.load(f)

with open(DATA_DIR / "dev-claims.json", "r") as f:
    dev_claims = json.load(f)

with open(DATA_DIR / "test-claims-unlabelled.json", "r") as f:
    test_claims = json.load(f)

with open(DATA_DIR / "evidence.json", "r") as f:
    evidence = json.load(f)

print("Train claims:", len(train_claims))
print("Dev claims:", len(dev_claims))
print("Test claims:", len(test_claims))
print("Evidence:", len(evidence))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train claims: 1228
Dev claims: 154
Test claims: 153
Evidence: 1208827


In [ ]:
# ============================================================
# Cell 4: Load query rewrite model
# ============================================================

REWRITE_MODEL_NAME = "prhegde/t5-query-reformulation-RL"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

rewrite_tokenizer = AutoTokenizer.from_pretrained(REWRITE_MODEL_NAME)
rewrite_model = AutoModelForSeq2SeqLM.from_pretrained(REWRITE_MODEL_NAME)

rewrite_model = rewrite_model.to(device)
rewrite_model.eval()

print("Loaded rewrite model:", REWRITE_MODEL_NAME)

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Loaded rewrite model: prhegde/t5-query-reformulation-RL


In [ ]:
# ============================================================
# Cell 5: Rewrite one claim for BGE retrieval
# ============================================================

def rewrite_claim_for_bge(
    claim_text,
    max_input_length=128,
    max_output_length=96,
    num_beams=4
):
    """
    Rewrite one claim into a retrieval-friendly query.

    Important:
    - claim_id is not changed
    - original claim_text is not changed
    - this rewritten text is only for BGE query encoding
    """

    claim_text = str(claim_text).strip()

    if claim_text == "":
        return claim_text

    inputs = rewrite_tokenizer(
        claim_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length
    ).to(device)

    with torch.no_grad():
        output_ids = rewrite_model.generate(
            **inputs,
            max_length=max_output_length,
            num_beams=num_beams,
            do_sample=False,
            early_stopping=True
        )

    rewritten = rewrite_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()

    if rewritten == "":
        rewritten = claim_text

    return rewritten

In [ ]:
# ============================================================
# Cell 6: Test rewriting one dev claim
# ============================================================

sample_cid = list(dev_claims.keys())[0]
sample_claim = dev_claims[sample_cid]["claim_text"]

sample_rewrite = rewrite_claim_for_bge(sample_claim)

print("Claim ID:")
print(sample_cid)

print("\nOriginal claim:")
print(sample_claim)

print("\nRewritten claim for BGE:")
print(sample_rewrite)

Claim ID:
claim-752

Original claim:
[South Australia] has the most expensive electricity in the world.

Rewritten claim for BGE:
most expensive electricity in south australia


In [ ]:
# ============================================================
# Cell 7: Rewrite all dev claims with cache
# ============================================================

SAVE_DIR = Path("/content/drive/MyDrive/NLP ASS3/rewrite_cache")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DEV_REWRITE_CACHE_PATH = SAVE_DIR / "dev_claims_rewritten_for_bge.pkl"

if DEV_REWRITE_CACHE_PATH.exists():
    with open(DEV_REWRITE_CACHE_PATH, "rb") as f:
        dev_claims_rewritten = pickle.load(f)
    print("Loaded cached dev rewrites:", DEV_REWRITE_CACHE_PATH)

else:
    dev_claims_rewritten = {}

    for cid, item in tqdm(dev_claims.items(), desc="Rewriting dev claims"):
        original_claim = item["claim_text"]
        rewritten_claim = rewrite_claim_for_bge(original_claim)

        dev_claims_rewritten[cid] = {
            "claim_text": original_claim,
            "rewritten_claim_text": rewritten_claim
        }

    with open(DEV_REWRITE_CACHE_PATH, "wb") as f:
        pickle.dump(dev_claims_rewritten, f)

    print("Saved dev rewrites:", DEV_REWRITE_CACHE_PATH)

print("Number of rewritten dev claims:", len(dev_claims_rewritten))

Rewriting dev claims:   0%|          | 0/154 [00:00<?, ?it/s]

Saved dev rewrites: /content/drive/MyDrive/NLP ASS3/rewrite_cache/dev_claims_rewritten_for_bge.pkl
Number of rewritten dev claims: 154


In [ ]:
# ============================================================
# Cell 8: Check claim_id alignment
# ============================================================

original_dev_ids = list(dev_claims.keys())
rewritten_dev_ids = list(dev_claims_rewritten.keys())

print("Same number:", len(original_dev_ids) == len(rewritten_dev_ids))
print("Same order:", original_dev_ids == rewritten_dev_ids)
print("Same ID set:", set(original_dev_ids) == set(rewritten_dev_ids))

example_id = original_dev_ids[0]

print("\nExample ID:", example_id)
print("\nOriginal:")
print(dev_claims[example_id]["claim_text"])
print("\nRewritten:")
print(dev_claims_rewritten[example_id]["rewritten_claim_text"])

Same number: True
Same order: True
Same ID set: True

Example ID: claim-752

Original:
[South Australia] has the most expensive electricity in the world.

Rewritten:
most expensive electricity in south australia


In [ ]:
# ============================================================
# Cell 9: Build BGE inputs using rewritten claims
# ============================================================

BGE_QUERY_PREFIX = "Represent this question for retrieving supporting evidence passages: "

evidence_ids = list(evidence.keys())
evidence_texts = [evidence[eid] for eid in evidence_ids]

dev_ids = list(dev_claims.keys())

dev_original_texts = [
    dev_claims[cid]["claim_text"]
    for cid in dev_ids
]

dev_rewritten_texts = [
    dev_claims_rewritten[cid]["rewritten_claim_text"]
    for cid in dev_ids
]

# Evidence 不改
bge_evidence_inputs = evidence_texts

# BGE query 用 rewritten claim
bge_dev_inputs = [
    BGE_QUERY_PREFIX + text
    for text in dev_rewritten_texts
]

print("Num evidence inputs:", len(bge_evidence_inputs))
print("Num dev inputs:", len(bge_dev_inputs))

print("\nExample claim_id:")
print(dev_ids[0])

print("\nOriginal claim:")
print(dev_original_texts[0])

print("\nRewritten claim:")
print(dev_rewritten_texts[0])

print("\nFinal BGE input:")
print(bge_dev_inputs[0])

Num evidence inputs: 1208827
Num dev inputs: 154

Example claim_id:
claim-752

Original claim:
[South Australia] has the most expensive electricity in the world.

Rewritten claim:
most expensive electricity in south australia

Final BGE input:
Represent this question for retrieving supporting evidence passages: most expensive electricity in south australia


In [ ]:
# ============================================================
# Cell 10: Load BGE model
# ============================================================

from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

bge_tokenizer = AutoTokenizer.from_pretrained(BGE_MODEL_NAME)
bge_model = AutoModel.from_pretrained(BGE_MODEL_NAME).to(device)
bge_model.eval()

print("Loaded BGE model:", BGE_MODEL_NAME)

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded BGE model: BAAI/bge-small-en-v1.5


In [ ]:
# ============================================================
# Cell 11: BGE encoding function
# ============================================================

def encode_bge_texts(texts, tokenizer, model, batch_size=64, max_len=256):
    all_embeddings = []

    with torch.no_grad():
        for start in tqdm(range(0, len(texts), batch_size), desc="Encoding with BGE"):
            batch_texts = texts[start:start + batch_size]

            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt"
            )

            encoded = {
                k: v.to(device)
                for k, v in encoded.items()
            }

            outputs = model(**encoded)

            # CLS pooling
            embeddings = outputs.last_hidden_state[:, 0]

            # normalize so dot product = cosine similarity
            embeddings = F.normalize(embeddings, p=2, dim=1)

            all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings, dim=0)

In [ ]:
# ============================================================
# Cell 12: Encode evidence and rewritten dev claims
# ============================================================

bge_evidence_emb = encode_bge_texts(
    bge_evidence_inputs,
    bge_tokenizer,
    bge_model,
    batch_size=64,
    max_len=256
)

bge_dev_emb = encode_bge_texts(
    bge_dev_inputs,
    bge_tokenizer,
    bge_model,
    batch_size=64,
    max_len=256
)

print("BGE evidence emb:", bge_evidence_emb.shape)
print("BGE dev emb:", bge_dev_emb.shape)

Encoding with BGE:   0%|          | 0/18888 [00:00<?, ?it/s]

Encoding with BGE:   0%|          | 0/3 [00:00<?, ?it/s]

BGE evidence emb: torch.Size([1208827, 384])
BGE dev emb: torch.Size([154, 384])


In [ ]:
# ============================================================
# Cell 13: Save rewritten BGE embeddings
# ============================================================

BGE_SAVE_DIR = Path("/content/drive/MyDrive/NLP ASS3/saved_bge_rewritten_embeddings")
BGE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

torch.save(bge_evidence_emb, BGE_SAVE_DIR / "bge_evidence_emb.pt")
torch.save(bge_dev_emb, BGE_SAVE_DIR / "bge_dev_rewritten_emb.pt")

with open(BGE_SAVE_DIR / "evidence_ids.pkl", "wb") as f:
    pickle.dump(evidence_ids, f)

with open(BGE_SAVE_DIR / "dev_ids.pkl", "wb") as f:
    pickle.dump(dev_ids, f)

with open(BGE_SAVE_DIR / "dev_claims_rewritten.pkl", "wb") as f:
    pickle.dump(dev_claims_rewritten, f)

print("Saved rewritten BGE embeddings to:", BGE_SAVE_DIR)

Saved rewritten BGE embeddings to: /content/drive/MyDrive/NLP ASS3/saved_bge_rewritten_embeddings


In [ ]:
# ============================================================
# Cell 14: Dense retrieval from BGE embeddings
# ============================================================

def dense_retrieve_from_embeddings(
    claim_ids,
    claim_emb,
    evidence_ids,
    evidence_emb,
    top_k=100,
    batch_size=32
):
    """
    claim_emb and evidence_emb are normalized.
    Dot product = cosine similarity.
    """

    predict = {}
    evidence_emb_t = evidence_emb.T

    for start in tqdm(range(0, len(claim_ids), batch_size), desc="Dense retrieving"):
        end = start + batch_size

        batch_claim_emb = claim_emb[start:end]
        scores = batch_claim_emb @ evidence_emb_t

        top_scores, top_indices = torch.topk(
            scores,
            k=top_k,
            dim=1
        )

        top_indices = top_indices.cpu().numpy()

        for i, cid in enumerate(claim_ids[start:end]):
            predict[cid] = [
                evidence_ids[idx]
                for idx in top_indices[i]
            ]

    return predict

In [ ]:
# ============================================================
# Cell 15: Retrieve top-k with rewritten BGE
# ============================================================

TOP_K = 5

predict_bge_rewritten_top5 = dense_retrieve_from_embeddings(
    claim_ids=dev_ids,
    claim_emb=bge_dev_emb,
    evidence_ids=evidence_ids,
    evidence_emb=bge_evidence_emb,
    top_k=TOP_K,
    batch_size=32
)

print("Example prediction:")
example_id = dev_ids[0]
print(example_id)
print(predict_bge_rewritten_top5[example_id])

Dense retrieving:   0%|          | 0/5 [00:00<?, ?it/s]

Example prediction:
claim-752
['evidence-67732', 'evidence-572512', 'evidence-932540', 'evidence-554677', 'evidence-452156']


In [ ]:
# ============================================================
# Cell 16: Evaluate rewritten BGE retrieval
# ============================================================
import numpy as np
eval_bge_rewritten_top5 = eval_retrieval(
    dev_claims,
    predict_bge_rewritten_top5
)

Mean Recall:    0.179870
Mean Precision: 0.090909
Mean F1-Score:  0.111663
Min Recall:     0.000000


# soft prompt embeddings + claim tokens → BGE encoder → query embedding
# BGE parameters + soft prompt parameters

In [1]:
# ============================================================
# New Cell 1: Install packages
# ============================================================

!pip -q install transformers sentencepiece accelerate tqdm

In [2]:
# ============================================================
# New Cell 2: Imports and device
# ============================================================

import os
import json
import pickle
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Device: cuda


In [3]:
# ============================================================
# New Cell 3: Load data
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path("/content/drive/MyDrive/NLP ASS3/data")

with open(DATA_DIR / "train-claims.json", "r") as f:
    train_claims = json.load(f)

with open(DATA_DIR / "dev-claims.json", "r") as f:
    dev_claims = json.load(f)

with open(DATA_DIR / "test-claims-unlabelled.json", "r") as f:
    test_claims = json.load(f)

with open(DATA_DIR / "evidence.json", "r") as f:
    evidence = json.load(f)

print("Train claims:", len(train_claims))
print("Dev claims:", len(dev_claims))
print("Test claims:", len(test_claims))
print("Evidence:", len(evidence))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train claims: 1228
Dev claims: 154
Test claims: 153
Evidence: 1208827


In [4]:
# ============================================================
# New Cell 4: Build positive training pairs
# ============================================================

train_pairs = []

for cid, item in train_claims.items():
    claim_text = item["claim_text"]
    gold_eids = item["evidences"]

    for eid in gold_eids:
        if eid in evidence:
            train_pairs.append({
                "claim_id": cid,
                "claim_text": claim_text,
                "evidence_id": eid,
                "evidence_text": evidence[eid]
            })

print("Number of positive train pairs:", len(train_pairs))

print("\nExample:")
print("Claim ID:", train_pairs[0]["claim_id"])
print("Claim:", train_pairs[0]["claim_text"])
print("Evidence ID:", train_pairs[0]["evidence_id"])
print("Evidence:", train_pairs[0]["evidence_text"])

Number of positive train pairs: 4122

Example:
Claim ID: claim-1937
Claim: Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.
Evidence ID: evidence-442946
Evidence: At very high concentrations (100 times atmospheric concentration, or greater), carbon dioxide can be toxic to animal life, so raising the concentration to 10,000 ppm (1%) or higher for several hours will eliminate pests such as whiteflies and spider mites in a greenhouse.


In [6]:
# ============================================================
# New Cell 5: Dataset and DataLoader
# ============================================================

class ClaimEvidenceDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        item = self.pairs[idx]
        return {
            "claim_text": item["claim_text"],
            "evidence_text": item["evidence_text"],
            "claim_id": item["claim_id"],
            "evidence_id": item["evidence_id"]
        }


def collate_fn(batch):
    return {
        "claim_texts": [x["claim_text"] for x in batch],
        "evidence_texts": [x["evidence_text"] for x in batch],
        "claim_ids": [x["claim_id"] for x in batch],
        "evidence_ids": [x["evidence_id"] for x in batch]
    }


train_dataset = ClaimEvidenceDataset(train_pairs)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
    drop_last=True
)

print("Train batches:", len(train_loader))

Train batches: 257


In [7]:
# ============================================================
# New Cell 6: BGE encoder with trainable soft prompt
# ============================================================

class BGEWithSoftPrompt(nn.Module):
    def __init__(
        self,
        model_name="BAAI/bge-small-en-v1.5",
        soft_prompt_len=8,
        init_std=0.02
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.encoder.config.hidden_size
        self.soft_prompt_len = soft_prompt_len

        self.soft_prompt = nn.Parameter(
            torch.randn(soft_prompt_len, self.hidden_size) * init_std
        )

    def mean_pooling(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = torch.sum(last_hidden_state * mask, dim=1)
        counted = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / counted

    def encode_query_with_soft_prompt(self, input_ids, attention_mask):
        """
        Query side:
        soft prompt + claim tokens
        """

        batch_size = input_ids.size(0)

        token_embeds = self.encoder.embeddings.word_embeddings(input_ids)

        soft_prompt = self.soft_prompt.unsqueeze(0).expand(batch_size, -1, -1)

        inputs_embeds = torch.cat(
            [soft_prompt, token_embeds],
            dim=1
        )

        soft_attention = torch.ones(
            batch_size,
            self.soft_prompt_len,
            dtype=attention_mask.dtype,
            device=attention_mask.device
        )

        extended_attention_mask = torch.cat(
            [soft_attention, attention_mask],
            dim=1
        )

        outputs = self.encoder(
            inputs_embeds=inputs_embeds,
            attention_mask=extended_attention_mask
        )

        # 用 CLS 位置。
        # 注意：加了 soft prompt 后，第 0 位是 soft prompt，不再是原本 CLS。
        # 所以这里取 soft prompt 之后的第一个 token，也就是 index = soft_prompt_len。
        query_emb = outputs.last_hidden_state[:, self.soft_prompt_len]

        query_emb = F.normalize(query_emb, p=2, dim=1)

        return query_emb

    def encode_passage(self, input_ids, attention_mask):
        """
        Evidence side:
        不加 soft prompt
        """

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        passage_emb = outputs.last_hidden_state[:, 0]
        passage_emb = F.normalize(passage_emb, p=2, dim=1)

        return passage_emb

In [8]:
# ============================================================
# New Cell 7: Load tokenizer and model
# ============================================================

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

bge_tokenizer = AutoTokenizer.from_pretrained(BGE_MODEL_NAME)

soft_prompt_bge = BGEWithSoftPrompt(
    model_name=BGE_MODEL_NAME,
    soft_prompt_len=8
).to(device)

print("Loaded:", BGE_MODEL_NAME)
print("Soft prompt length:", soft_prompt_bge.soft_prompt_len)
print("Hidden size:", soft_prompt_bge.hidden_size)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded: BAAI/bge-small-en-v1.5
Soft prompt length: 8
Hidden size: 384


In [9]:
# ============================================================
# New Cell 8: Tokenization helper
# ============================================================

def tokenize_texts(texts, tokenizer, max_length=256):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

In [10]:
# ============================================================
# New Cell 9: Training setup
# ============================================================

EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

optimizer = torch.optim.AdamW(
    soft_prompt_bge.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

num_training_steps = len(train_loader) * EPOCHS
num_warmup_steps = int(num_training_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

print("Training steps:", num_training_steps)
print("Warmup steps:", num_warmup_steps)

Training steps: 771
Warmup steps: 77


在cell10做的soft prompt tuning

In [11]:
# ============================================================
# New Cell 10: Train BGE + soft prompt using contrastive loss
# ============================================================

def train_one_epoch(model, train_loader, tokenizer, optimizer, scheduler, epoch):
    model.train()

    total_loss = 0.0

    progress = tqdm(train_loader, desc=f"Epoch {epoch}")

    for batch in progress:
        claim_texts = batch["claim_texts"]
        evidence_texts = batch["evidence_texts"]

        claim_tokens = tokenize_texts(
            claim_texts,
            tokenizer,
            max_length=128
        )

        evidence_tokens = tokenize_texts(
            evidence_texts,
            tokenizer,
            max_length=256
        )

        claim_tokens = {
            k: v.to(device)
            for k, v in claim_tokens.items()
        }

        evidence_tokens = {
            k: v.to(device)
            for k, v in evidence_tokens.items()
        }

        query_emb = model.encode_query_with_soft_prompt(
            input_ids=claim_tokens["input_ids"],
            attention_mask=claim_tokens["attention_mask"]
        )

        passage_emb = model.encode_passage(
            input_ids=evidence_tokens["input_ids"],
            attention_mask=evidence_tokens["attention_mask"]
        )

        scores = query_emb @ passage_emb.T

        temperature = 0.05
        scores = scores / temperature

        labels = torch.arange(
            scores.size(0),
            device=scores.device
        )

        loss_q_to_p = F.cross_entropy(scores, labels)
        loss_p_to_q = F.cross_entropy(scores.T, labels)

        loss = (loss_q_to_p + loss_p_to_q) / 2

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        progress.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    avg_loss = total_loss / len(train_loader)
    return avg_loss


for epoch in range(1, EPOCHS + 1):
    avg_loss = train_one_epoch(
        model=soft_prompt_bge,
        train_loader=train_loader,
        tokenizer=bge_tokenizer,
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=epoch
    )

    print(f"Epoch {epoch} average loss: {avg_loss:.4f}")

Epoch 1:   0%|          | 0/257 [00:00<?, ?it/s]

Epoch 1 average loss: 0.7447


Epoch 2:   0%|          | 0/257 [00:00<?, ?it/s]

Epoch 2 average loss: 0.4505


Epoch 3:   0%|          | 0/257 [00:00<?, ?it/s]

Epoch 3 average loss: 0.3656


In [12]:
# ============================================================
# New Cell 11: Save trained BGE + soft prompt
# ============================================================

SAVE_DIR = Path("/content/drive/MyDrive/NLP ASS3/bge_soft_prompt_finetuned")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_state_dict": soft_prompt_bge.state_dict(),
        "soft_prompt_len": soft_prompt_bge.soft_prompt_len,
        "model_name": BGE_MODEL_NAME
    },
    SAVE_DIR / "bge_soft_prompt_model.pt"
)

bge_tokenizer.save_pretrained(SAVE_DIR / "tokenizer")

print("Saved model to:", SAVE_DIR)

Saved model to: /content/drive/MyDrive/NLP ASS3/bge_soft_prompt_finetuned


In [13]:
# ============================================================
# New Cell 12: Encoding functions for retrieval
# ============================================================

def encode_queries_soft_prompt(
    model,
    tokenizer,
    texts,
    batch_size=64,
    max_length=128
):
    model.eval()
    all_embs = []

    with torch.no_grad():
        for start in tqdm(range(0, len(texts), batch_size), desc="Encoding queries"):
            batch_texts = texts[start:start + batch_size]

            tokens = tokenize_texts(
                batch_texts,
                tokenizer,
                max_length=max_length
            )

            tokens = {
                k: v.to(device)
                for k, v in tokens.items()
            }

            emb = model.encode_query_with_soft_prompt(
                input_ids=tokens["input_ids"],
                attention_mask=tokens["attention_mask"]
            )

            all_embs.append(emb.cpu())

    return torch.cat(all_embs, dim=0)


def encode_passages_no_prompt(
    model,
    tokenizer,
    texts,
    batch_size=64,
    max_length=256
):
    model.eval()
    all_embs = []

    with torch.no_grad():
        for start in tqdm(range(0, len(texts), batch_size), desc="Encoding evidences"):
            batch_texts = texts[start:start + batch_size]

            tokens = tokenize_texts(
                batch_texts,
                tokenizer,
                max_length=max_length
            )

            tokens = {
                k: v.to(device)
                for k, v in tokens.items()
            }

            emb = model.encode_passage(
                input_ids=tokens["input_ids"],
                attention_mask=tokens["attention_mask"]
            )

            all_embs.append(emb.cpu())

    return torch.cat(all_embs, dim=0)

In [14]:
# ============================================================
# New Cell 13: Encode dev claims and all evidence
# ============================================================

evidence_ids = list(evidence.keys())
evidence_texts = [evidence[eid] for eid in evidence_ids]

dev_ids = list(dev_claims.keys())
dev_texts = [
    dev_claims[cid]["claim_text"]
    for cid in dev_ids
]

bge_soft_evidence_emb = encode_passages_no_prompt(
    model=soft_prompt_bge,
    tokenizer=bge_tokenizer,
    texts=evidence_texts,
    batch_size=64,
    max_length=256
)

bge_soft_dev_emb = encode_queries_soft_prompt(
    model=soft_prompt_bge,
    tokenizer=bge_tokenizer,
    texts=dev_texts,
    batch_size=64,
    max_length=128
)

print("Evidence emb:", bge_soft_evidence_emb.shape)
print("Dev emb:", bge_soft_dev_emb.shape)

Encoding evidences:   0%|          | 0/18888 [00:00<?, ?it/s]

Encoding queries:   0%|          | 0/3 [00:00<?, ?it/s]

Evidence emb: torch.Size([1208827, 384])
Dev emb: torch.Size([154, 384])


In [15]:
# ============================================================
# New Cell 14: Dense retrieval
# ============================================================

def dense_retrieve_from_embeddings(
    claim_ids,
    claim_emb,
    evidence_ids,
    evidence_emb,
    top_k=5,
    batch_size=32
):
    predict = {}

    evidence_emb_t = evidence_emb.T

    for start in tqdm(range(0, len(claim_ids), batch_size), desc="Dense retrieving"):
        end = start + batch_size

        batch_claim_emb = claim_emb[start:end]
        scores = batch_claim_emb @ evidence_emb_t

        top_scores, top_indices = torch.topk(
            scores,
            k=top_k,
            dim=1
        )

        top_indices = top_indices.cpu().numpy()

        for i, cid in enumerate(claim_ids[start:end]):
            predict[cid] = [
                evidence_ids[idx]
                for idx in top_indices[i]
            ]

    return predict

In [18]:
# ============================================================
# New Cell 15: Evaluate fine-tuned BGE + soft prompt top5
# ============================================================

predict_bge_soft_prompt_top5 = dense_retrieve_from_embeddings(
    claim_ids=dev_ids,
    claim_emb=bge_soft_dev_emb,
    evidence_ids=evidence_ids,
    evidence_emb=bge_soft_evidence_emb,
    top_k=5,
    batch_size=32
)

print("Fine-tuned BGE + soft prompt top5")
eval_bge_soft_prompt_top5 = eval_retrieval(
    dev_claims,
    predict_bge_soft_prompt_top5
)

Dense retrieving:   0%|          | 0/5 [00:00<?, ?it/s]

Fine-tuned BGE + soft prompt top5
Mean Recall:    0.278355
Mean Precision: 0.158442
Mean F1-Score:  0.188039
Min Recall:     0.000000


In [19]:
# ============================================================
# New Cell 16: Evaluate fine-tuned BGE + soft prompt top100
# ============================================================

predict_bge_soft_prompt_top100 = dense_retrieve_from_embeddings(
    claim_ids=dev_ids,
    claim_emb=bge_soft_dev_emb,
    evidence_ids=evidence_ids,
    evidence_emb=bge_soft_evidence_emb,
    top_k=100,
    batch_size=32
)

print("Fine-tuned BGE + soft prompt top100")
eval_bge_soft_prompt_top100 = eval_retrieval(
    dev_claims,
    predict_bge_soft_prompt_top100
)

Dense retrieving:   0%|          | 0/5 [00:00<?, ?it/s]

Fine-tuned BGE + soft prompt top100
Mean Recall:    0.699242
Mean Precision: 0.021364
Mean F1-Score:  0.041158
Min Recall:     0.000000


In [30]:
CANDIDATE_K = 100

print("Retrieving BM25 top-100 candidates...")
predict_R2_bm25_top100 = bm25s_retrieve(
    dev_claims,
    k=CANDIDATE_K
)

Retrieving BM25 top-100 candidates...


Split strings:   0%|          | 0/154 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/154 [00:00<?, ?it/s]

In [31]:
# ============================================================
# New Cell 18: RRF with BM25 + fine-tuned BGE soft prompt
# ============================================================

def reciprocal_rank_fusion(
    pred_a,
    pred_b,
    weight_a=1.0,
    weight_b=1.0,
    k=60,
    final_k=5
):
    fused = {}

    for cid in pred_a.keys():
        scores = {}

        for rank, eid in enumerate(pred_a[cid]):
            scores[eid] = scores.get(eid, 0.0) + weight_a * (1.0 / (k + rank + 1))

        for rank, eid in enumerate(pred_b[cid]):
            scores[eid] = scores.get(eid, 0.0) + weight_b * (1.0 / (k + rank + 1))

        ranked = sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )

        fused[cid] = [
            eid
            for eid, score in ranked[:final_k]
        ]

    return fused


rrf_settings = []

for bm25_w in [0.5, 0.7, 1.0, 1.2]:
    for bge_w in [1.0, 1.5, 2.0, 3.0, 4.0]:
        rrf_settings.append((bm25_w, bge_w))


best_soft_rrf = None
best_soft_rrf_score = None
best_soft_rrf_f1 = -1
best_soft_rrf_weights = None

for bm25_w, bge_w in rrf_settings:
    pred = reciprocal_rank_fusion(
        pred_a=predict_R2_bm25_top100,
        pred_b=predict_bge_soft_prompt_top100,
        weight_a=bm25_w,
        weight_b=bge_w,
        k=60,
        final_k=5
    )

    print(f"\nRRF setting: BM25 weight={bm25_w}, Soft-BGE weight={bge_w}")

    score = eval_retrieval(
        dev_claims,
        pred
    )

    f1, precision, recall, min_recall = score

    if f1 > best_soft_rrf_f1:
        best_soft_rrf_f1 = f1
        best_soft_rrf_score = score
        best_soft_rrf = pred
        best_soft_rrf_weights = (bm25_w, bge_w)


print("\nBest RRF weights:", best_soft_rrf_weights)
print("Best RRF score:", best_soft_rrf_score)


RRF setting: BM25 weight=0.5, Soft-BGE weight=1.0
Mean Recall:    0.293506
Mean Precision: 0.164935
Mean F1-Score:  0.195872
Min Recall:     0.000000

RRF setting: BM25 weight=0.5, Soft-BGE weight=1.5
Mean Recall:    0.298377
Mean Precision: 0.166234
Mean F1-Score:  0.197624
Min Recall:     0.000000

RRF setting: BM25 weight=0.5, Soft-BGE weight=2.0
Mean Recall:    0.295779
Mean Precision: 0.163636
Mean F1-Score:  0.195027
Min Recall:     0.000000

RRF setting: BM25 weight=0.5, Soft-BGE weight=3.0
Mean Recall:    0.292208
Mean Precision: 0.163636
Mean F1-Score:  0.194444
Min Recall:     0.000000

RRF setting: BM25 weight=0.5, Soft-BGE weight=4.0
Mean Recall:    0.298701
Mean Precision: 0.166234
Mean F1-Score:  0.198155
Min Recall:     0.000000

RRF setting: BM25 weight=0.7, Soft-BGE weight=1.0
Mean Recall:    0.290801
Mean Precision: 0.166234
Mean F1-Score:  0.196774
Min Recall:     0.000000

RRF setting: BM25 weight=0.7, Soft-BGE weight=1.5
Mean Recall:    0.292208
Mean Precision: 0.

In [34]:
# ============================================================
# New Cell 19: Compare retrieval results
# ============================================================

comparison_results = []

def add_result(name, score_tuple):
    f1, precision, recall, min_recall = score_tuple
    comparison_results.append({
        "method": name,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "min_recall": min_recall
    })




add_result(
    "Fine-tuned BGE + Soft Prompt top5",
    eval_retrieval(dev_claims, predict_bge_soft_prompt_top5)
)

add_result(
    "BM25 + Fine-tuned Soft-BGE RRF top5",
    eval_retrieval(dev_claims, best_soft_rrf)
)


for row in comparison_results:
    print(row)

Mean Recall:    0.278355
Mean Precision: 0.158442
Mean F1-Score:  0.188039
Min Recall:     0.000000
Mean Recall:    0.301623
Mean Precision: 0.167532
Mean F1-Score:  0.199479
Min Recall:     0.000000
{'method': 'Fine-tuned BGE + Soft Prompt top5', 'f1': np.float64(0.1880385487528345), 'precision': np.float64(0.15844155844155844), 'recall': np.float64(0.27835497835497836), 'min_recall': 0.0}
{'method': 'BM25 + Fine-tuned Soft-BGE RRF top5', 'f1': np.float64(0.19947948876520305), 'precision': np.float64(0.16753246753246753), 'recall': np.float64(0.3016233766233766), 'min_recall': 0.0}
